# onnx export and runtime

goal: train in pytorch, export to onnx, run with onnxruntime. faster CPU inference than torch.

In [ ]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
    def forward(self, x):
        return self.fc(x)

m = Net().eval()
dummy = torch.randn(1, 10)
torch.onnx.export(m, dummy, 'tiny.onnx', input_names=['x'], output_names=['y'], dynamic_axes={'x': {0: 'batch'}, 'y': {0: 'batch'}}, opset_version=11)

## load + run with onnxruntime

In [ ]:
import onnxruntime as ort
import numpy as np
sess = ort.InferenceSession('tiny.onnx', providers=['CPUExecutionProvider'])
out = sess.run(None, {'x': np.random.randn(4, 10).astype(np.float32)})
out[0].shape

## benchmark torch vs ort

In [ ]:
import time
x_np = np.random.randn(64, 10).astype(np.float32)
x_t = torch.from_numpy(x_np)

t0 = time.time()
for _ in range(1000):
    with torch.no_grad():
        m(x_t)
print('torch:', time.time()-t0)

t0 = time.time()
for _ in range(1000):
    sess.run(None, {'x': x_np})
print('ort:', time.time()-t0)

In [ ]:
# re-ran with seed=42

note: tightened up
